In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/merged_dataset.csv")


In [3]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,cart,5773203,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
1,2019-10-01 00:00:03 UTC,cart,5773353,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
2,2019-10-01 00:00:07 UTC,cart,5881589,2151191071051219817,NaN,lovely,13.48,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9
3,2019-10-01 00:00:07 UTC,cart,5723490,1487580005134238553,NaN,runail,2.62,463240011,26dd6e6e-4dac-4778-8d2c-92e149dab885
4,2019-10-01 00:00:15 UTC,cart,5881449,1487580013522845895,NaN,lovely,0.56,429681830,49e8d843-adf3-428b-a2c3-fe8bc6a307c9


In [58]:
clean_df = df.copy()

# Null counts and percentages per column
null_report = pd.DataFrame({
    "null_count": clean_df.isnull().sum(),
    "null_pct": (clean_df.isnull().mean() * 100).round(2)
})
print(null_report)

# Duplicate rows
print("\nDuplicate rows:", clean_df.duplicated().sum())

# Price sanity check
print("\nPrice summary:")
print(clean_df["price"].describe())
print("Price <= 0:", (clean_df["price"] <= 0).sum())

               null_count  null_pct
event_time              0      0.00
event_type              0      0.00
product_id              0      0.00
category_id             0      0.00
category_code    12069716     98.36
brand             5155579     42.01
price                   0      0.00
user_id                 0      0.00
user_session         2229      0.02

Duplicate rows: 643708

Price summary:
count    1.227141e+07
mean     8.475452e+00
std      1.922233e+01
min     -7.937000e+01
25%      2.050000e+00
50%      4.000000e+00
75%      6.900000e+00
max      3.277800e+02
Name: price, dtype: float64
Price <= 0: 25335


In [59]:
# ---- category_code recoverability (via category_id) ----
has_code = clean_df.dropna(subset=["category_code"])

# How many category_ids have at least one known code
ids_with_code = has_code["category_id"].nunique()
ids_total = clean_df["category_id"].nunique()
print("category_id with a known code:", ids_with_code, "/", ids_total)

# Is the mapping consistent? (one category_id should map to one code)
codes_per_id = has_code.groupby("category_id")["category_code"].nunique()
print("category_id with more than 1 code:", (codes_per_id > 1).sum())

# How many null rows can be recovered through this mapping
null_code_rows = clean_df["category_code"].isna()
recoverable_code = null_code_rows & clean_df["category_id"].isin(has_code["category_id"].unique())
print("Null category_code rows:", null_code_rows.sum())
print("Recoverable category_code rows:", recoverable_code.sum())

# ---- brand recoverability (via product_id) ----
has_brand = clean_df.dropna(subset=["brand"])

products_with_brand = has_brand["product_id"].nunique()
products_total = clean_df["product_id"].nunique()
print("\nproduct_id with a known brand:", products_with_brand, "/", products_total)

# Is the mapping consistent? (one product_id should map to one brand)
brands_per_product = has_brand.groupby("product_id")["brand"].nunique()
print("product_id with more than 1 brand:", (brands_per_product > 1).sum())

# How many null rows can be recovered through this mapping
null_brand_rows = clean_df["brand"].isna()
recoverable_brand = null_brand_rows & clean_df["product_id"].isin(has_brand["product_id"].unique())
print("Null brand rows:", null_brand_rows.sum())
print("Recoverable brand rows:", recoverable_brand.sum())

category_id with a known code: 17 / 514
category_id with more than 1 code: 0
Null category_code rows: 12069716
Recoverable category_code rows: 0

product_id with a known brand: 28453 / 49341
product_id with more than 1 brand: 23
Null brand rows: 5155579
Recoverable brand rows: 51723


In [60]:
# Flag which rows originally had a null brand (before any filling)
clean_df["brand_was_null"] = clean_df["brand"].isna()

# Standardize brand text: strip spaces, lowercase, empty string -> missing
clean_df["brand"] = clean_df["brand"].astype("string").str.strip().str.lower()
clean_df["brand"] = clean_df["brand"].replace("", pd.NA)

# Build mapping: product_id -> most frequent non-null brand
brand_map = (
    clean_df.dropna(subset=["brand"])
    .groupby(["product_id", "brand"])
    .size()
    .reset_index(name="n")
    .sort_values(["product_id", "n"], ascending=[True, False])
    .drop_duplicates("product_id")
    .set_index("product_id")["brand"]
)

# Fill missing brands using the mapping
missing_mask = clean_df["brand"].isna()
clean_df.loc[missing_mask, "brand"] = clean_df.loc[missing_mask, "product_id"].map(brand_map)

print("Remaining null brand:", clean_df["brand"].isna().sum())
print("Remaining null brand %:", round(clean_df["brand"].isna().mean() * 100, 2))

# Top 10 brands among known values
top_brands = clean_df["brand"].value_counts().head(10)
print("\nTop 10 brands:")
print(top_brands)
print("\nTop 10 share of known brands (%):")
print((top_brands / clean_df["brand"].notna().sum() * 100).round(2))

Remaining null brand: 5103856
Remaining null brand %: 41.59

Top 10 brands:
brand
runail       888674
irisk        584214
masura       508297
grattol      491555
bpw.style    327675
ingarden     258759
estel        237849
kapous       187799
freedecor    152722
jessnail     152328
Name: count, dtype: int64[pyarrow]

Top 10 share of known brands (%):
brand
runail       12.4
irisk        8.15
masura       7.09
grattol      6.86
bpw.style    4.57
ingarden     3.61
estel        3.32
kapous       2.62
freedecor    2.13
jessnail     2.13
Name: count, dtype: double[pyarrow]


In [61]:
# Fixed seed so the result is reproducible
rng = np.random.default_rng(42)

# Top 10 brands and their probabilities (proportional to their own frequency)
top10 = clean_df["brand"].value_counts().head(10)
top10_names = top10.index.tolist()
top10_probs = (top10 / top10.sum()).to_numpy(dtype=float)

print("Fill probabilities:")
for name, p in zip(top10_names, top10_probs):
    print(f"  {name}: {p:.4f}")

# Fill only the rows that still have a null brand
null_mask = clean_df["brand"].isna()
n_null = int(null_mask.sum())

clean_df["brand_imputed"] = False
clean_df.loc[null_mask, "brand"] = rng.choice(top10_names, size=n_null, p=top10_probs)
clean_df.loc[null_mask, "brand_imputed"] = True

# Convert to category dtype to save memory
clean_df["brand"] = clean_df["brand"].astype("category")

# Verification
print("\nRemaining null brand:", clean_df["brand"].isna().sum())
print("Imputed rows:", clean_df["brand_imputed"].sum())
print("\nTop 10 brands after fill:")
print(clean_df["brand"].value_counts().head(10))

Fill probabilities:
  runail: 0.2345
  irisk: 0.1542
  masura: 0.1341
  grattol: 0.1297
  bpw.style: 0.0865
  ingarden: 0.0683
  estel: 0.0628
  kapous: 0.0496
  freedecor: 0.0403
  jessnail: 0.0402

Remaining null brand: 0
Imputed rows: 5103856

Top 10 brands after fill:
brand
runail       2085818
irisk        1370679
masura       1194099
grattol      1152851
bpw.style     768098
ingarden      607647
estel         558589
kapous        440015
freedecor     358129
jessnail      357803
Name: count, dtype: int64


In [62]:
# ---- Which category_ids have a known code, and how many rows each ----
known = clean_df.dropna(subset=["category_code"])
code_map = (
    known.groupby(["category_id", "category_code"])
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
print("Known category codes:")
print(code_map.to_string(index=False))

# ---- Top 15 category_ids overall (with and without a code) ----
cat_counts = clean_df["category_id"].value_counts().head(15)
print("\nTop 15 category_ids by rows:")
print(cat_counts)

# Share of rows covered by category_ids that have a known code
covered = clean_df["category_id"].isin(code_map["category_id"]).mean() * 100
print("\nRows whose category_id has a known code (%):", round(covered, 2))

# ---- Relation between REAL brands and category_id (imputed rows excluded) ----
real = clean_df[~clean_df["brand_imputed"]]
top_cats = real["category_id"].value_counts().head(5).index

for cat in top_cats:
    print(f"\ncategory_id {cat} -> top real brands:")
    print(real[real["category_id"] == cat]["brand"].value_counts().head(5))

Known category codes:
        category_id                          category_code  rows
1487580006350586771          appliances.environment.vacuum 85579
1487580013053083824                    stationery.cartrige 36222
2007399943458784057                          apparel.glove 26289
2193074740619379535          furniture.living_room.cabinet 17917
1487580010695884882                        accessories.bag 16619
1487580011970953351                furniture.bathroom.bath  5485
2193074740686488401                furniture.bathroom.bath  5055
1487580008070251489        appliances.personal.hair_cutter  2745
2018287324474901238                furniture.bathroom.bath  2043
1921723506584715388               accessories.cosmetic_bag  1941
1487580012147114126                furniture.bathroom.bath   503
1783999067181810204 appliances.environment.air_conditioner   364
2022622168218599898            furniture.living_room.chair   328
1487580012759482531                furniture.bathroom.bath   300
148

In [63]:
# ---- 1. Null report for all columns ----
null_report = pd.DataFrame({
    "null_count": clean_df.isnull().sum(),
    "null_pct": (clean_df.isnull().mean() * 100).round(2)
})
print(null_report)

# ---- 2. Flag columns ----
print("\nbrand_was_null rows:", clean_df["brand_was_null"].sum())
print("brand_imputed rows:", clean_df["brand_imputed"].sum())
print("Recovered from product_id:",
      clean_df["brand_was_null"].sum() - clean_df["brand_imputed"].sum())

# ---- 3. Brand vs category_id consistency of imputed rows ----
real = clean_df[~clean_df["brand_imputed"]]
real_pairs = (
    real[["category_id", "brand"]]
    .assign(brand=lambda d: d["brand"].astype(str))
    .drop_duplicates()
)
real_pairs["pair_seen"] = True

imputed = (
    clean_df.loc[clean_df["brand_imputed"], ["category_id", "brand"]]
    .assign(brand=lambda d: d["brand"].astype(str))
)
check = imputed.merge(real_pairs, on=["category_id", "brand"], how="left")
print("\nImputed rows whose brand is never seen in the same category_id (%):",
      round(check["pair_seen"].isna().mean() * 100, 2))

# ---- 4. Price and duplicates ----
print("\nPrice summary:")
print(clean_df["price"].describe())
print("Price <= 0:", (clean_df["price"] <= 0).sum())
print("Duplicate rows:", clean_df.duplicated().sum())

# ---- 5. Null user_session by event type ----
print("\nNull user_session by event_type:")
print(clean_df[clean_df["user_session"].isna()]["event_type"].value_counts())

                null_count  null_pct
event_time               0      0.00
event_type               0      0.00
product_id               0      0.00
category_id              0      0.00
category_code     12069716     98.36
brand                    0      0.00
price                    0      0.00
user_id                  0      0.00
user_session          2229      0.02
brand_was_null           0      0.00
brand_imputed            0      0.00

brand_was_null rows: 5155579
brand_imputed rows: 5103856
Recovered from product_id: 51723

Imputed rows whose brand is never seen in the same category_id (%): 74.52

Price summary:
count    1.227141e+07
mean     8.475452e+00
std      1.922233e+01
min     -7.937000e+01
25%      2.050000e+00
50%      4.000000e+00
75%      6.900000e+00
max      3.277800e+02
Name: price, dtype: float64
Price <= 0: 25335
Duplicate rows: 409046

Null user_session by event_type:
event_type
cart                1818
remove_from_cart     394
view                  17
Name: cou

In [64]:
# Save the known category_id -> category_code mapping before dropping the column
category_lookup = (
    clean_df.dropna(subset=["category_code"])
    [["category_id", "category_code"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
category_lookup.to_csv("category_lookup.csv", index=False)
print("Saved lookup rows:", len(category_lookup))

# Drop the category_code column
clean_df = clean_df.drop(columns=["category_code"])

# Drop rows with a null user_session
before = len(clean_df)
clean_df = clean_df.dropna(subset=["user_session"])
print("Removed null user_session rows:", before - len(clean_df))

# Drop rows with invalid (zero or negative) price
before = len(clean_df)
clean_df = clean_df[clean_df["price"] > 0]
print("Removed invalid price rows:", before - len(clean_df))

# Drop exact duplicate rows
before = len(clean_df)
clean_df = clean_df.drop_duplicates()
print("Removed duplicate rows:", before - len(clean_df))

clean_df = clean_df.reset_index(drop=True)

# Final checks
print("\nFinal shape:", clean_df.shape)
print(clean_df.isnull().sum())
print("Min price:", clean_df["price"].min())
print(clean_df["event_type"].value_counts())

Saved lookup rows: 17
Removed null user_session rows: 2229
Removed invalid price rows: 25335
Removed duplicate rows: 408774

Final shape: (11835068, 10)
event_time        0
event_type        0
product_id        0
category_id       0
brand             0
price             0
user_id           0
user_session      0
brand_was_null    0
brand_imputed     0
dtype: int64
Min price: 0.05
event_type
view                5646546
cart                3421733
remove_from_cart    1986101
purchase             780688
Name: count, dtype: int64


In [65]:
# Check whether the category-aware re-imputation has already been applied
print("brand_fallback exists:", "brand_fallback" in clean_df.columns)
print("Index is default range:", clean_df.index.equals(pd.RangeIndex(len(clean_df))))

brand_fallback exists: False
Index is default range: True


In [66]:
# Fixed seed for reproducibility
rng = np.random.default_rng(42)

# Work with brand as plain objects during re-imputation
clean_df["brand"] = clean_df["brand"].astype(object)

# Step 1: undo the previous random fill (product_id-recovered rows are kept)
undo_mask = clean_df["brand_imputed"].to_numpy()
clean_df.loc[undo_mask, "brand"] = np.nan

# Step 2: real brand distribution per category_id (imputed rows excluded)
real = clean_df.loc[~undo_mask]
dist = real.groupby(["category_id", "brand"]).size().reset_index(name="n")
dist["p"] = dist["n"] / dist.groupby("category_id")["n"].transform("sum")
dist_groups = {
    cat: (g["brand"].to_numpy(), g["p"].to_numpy(dtype=float))
    for cat, g in dist.groupby("category_id")
}

# Step 3: global fallback distribution (top 10 real brands, weighted)
top10 = real["brand"].value_counts().head(10)
fallback_names = top10.index.to_numpy()
fallback_probs = (top10 / top10.sum()).to_numpy(dtype=float)

# Step 4: positions of null rows in the FULL dataframe and their category_ids
null_positions = np.flatnonzero(clean_df["brand"].isna().to_numpy())
null_cats = clean_df["category_id"].to_numpy()[null_positions]

new_brands = clean_df["brand"].to_numpy(dtype=object).copy()
imputed_flag = np.zeros(len(clean_df), dtype=bool)
fallback_flag = np.zeros(len(clean_df), dtype=bool)

# Group the null rows by category_id (indices are relative to null_positions)
groups = pd.Series(np.arange(len(null_positions))).groupby(null_cats).indices

for cat, idx in groups.items():
    pos = null_positions[idx]
    if cat in dist_groups:
        names, probs = dist_groups[cat]
        is_fallback = False
    else:
        names, probs = fallback_names, fallback_probs
        is_fallback = True
    new_brands[pos] = rng.choice(names, size=len(pos), p=probs)
    imputed_flag[pos] = True
    fallback_flag[pos] = is_fallback

clean_df["brand"] = new_brands
clean_df["brand_imputed"] = imputed_flag
clean_df["brand_fallback"] = fallback_flag
clean_df["brand"] = clean_df["brand"].astype("category")

# Verification
print("Remaining null brand:", clean_df["brand"].isna().sum())
print("Imputed rows:", clean_df["brand_imputed"].sum())
print("Fallback rows (category had no real brand):", clean_df["brand_fallback"].sum())

# Mismatch check: imputed (non-fallback) rows whose brand is unseen in the same category_id
real_pairs = (
    clean_df.loc[~clean_df["brand_imputed"], ["category_id", "brand"]]
    .astype({"brand": str})
    .drop_duplicates()
    .assign(pair_seen=True)
)
chk = (
    clean_df.loc[clean_df["brand_imputed"] & ~clean_df["brand_fallback"], ["category_id", "brand"]]
    .astype({"brand": str})
    .merge(real_pairs, on=["category_id", "brand"], how="left")
)
print("Mismatch % (non-fallback imputed rows):", round(chk["pair_seen"].isna().mean() * 100, 2))

Remaining null brand: 0
Imputed rows: 5049205
Fallback rows (category had no real brand): 1618873
Mismatch % (non-fallback imputed rows): 0.0


In [67]:
# Switch: True -> fallback rows get "unknown", False -> keep the random top-10 brand
FALLBACK_TO_UNKNOWN = True

# Drop the redundant flag (nearly identical to brand_imputed)
clean_df = clean_df.drop(columns=["brand_was_null"])

# Handle fallback rows
if FALLBACK_TO_UNKNOWN:
    brand_obj = clean_df["brand"].astype(object)
    brand_obj[clean_df["brand_fallback"].to_numpy()] = "unknown"
    clean_df["brand"] = brand_obj.astype("category")

# Memory-friendly dtypes
clean_df["event_type"] = clean_df["event_type"].astype("category")

# Final checks
print("Final shape:", clean_df.shape)
print(clean_df.dtypes)
print("\nNull counts:")
print(clean_df.isnull().sum())

print("\nBrand share among rows with a real or category-aware brand (%):")
reliable = clean_df[~clean_df["brand_fallback"]]
print((reliable["brand"].value_counts(normalize=True).head(10) * 100).round(2))

print("\nRows by brand source:")
print("Real brand:", (~clean_df["brand_imputed"]).sum())
print("Imputed (category-aware):", (clean_df["brand_imputed"] & ~clean_df["brand_fallback"]).sum())
print("Fallback:", clean_df["brand_fallback"].sum())

# Save the cleaned dataset
clean_df.to_parquet("clean_events.parquet", index=False)
print("\nSaved: clean_events.parquet")

Final shape: (11835068, 10)
event_time             str
event_type        category
product_id           int64
category_id          int64
brand             category
price              float64
user_id              int64
user_session           str
brand_imputed         bool
brand_fallback        bool
dtype: object

Null counts:
event_time        0
event_type        0
product_id        0
category_id       0
brand             0
price             0
user_id           0
user_session      0
brand_imputed     0
brand_fallback    0
dtype: int64

Brand share among rows with a real or category-aware brand (%):
brand
runail       12.85
irisk        10.53
grattol       5.47
masura        5.08
bpw.style     3.97
ingarden      3.41
estel         3.16
kapous        2.40
uno           2.30
jessnail      2.04
Name: proportion, dtype: float64

Rows by brand source:
Real brand: 6785863
Imputed (category-aware): 3430332
Fallback: 1618873

Saved: clean_events.parquet


In [68]:
# ---- 1. Inspect the raw strings before converting ----
raw = clean_df["event_time"].astype(str)
print("Sample raw values:")
print(raw.head(3).tolist())

# Check that every value matches the expected pattern: YYYY-MM-DD HH:MM:SS UTC
pattern_ok = raw.str.match(r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} UTC$")
print("\nValues NOT matching the expected format:", (~pattern_ok).sum())

# ---- 2. Convert with an explicit format (strict, no silent guessing) ----
converted = pd.to_datetime(
    raw,
    format="%Y-%m-%d %H:%M:%S UTC",
    utc=True,
    errors="coerce"
)

# ---- 3. Verify the conversion ----
print("\nDtype after conversion:", converted.dtype)
print("Null after conversion:", converted.isna().sum())
print("Date range:", converted.min(), "->", converted.max())

# Rows outside October 2019 (the dataset is expected to be one month)
outside = ~converted.dt.strftime("%Y-%m").eq("2019-10")
print("Rows outside 2019-10:", outside.sum())

# Distribution by day and hour
print("\nRows per day (first 5 days):")
print(converted.dt.date.value_counts().sort_index().head(5))
print("\nRows per hour of day:")
print(converted.dt.hour.value_counts().sort_index())

# Is the data sorted by time?
print("\nSorted by event_time:", converted.is_monotonic_increasing)

# ---- 4. Apply only if there are no nulls ----
if converted.isna().sum() == 0:
    clean_df["event_time"] = converted
    print("\nevent_time converted successfully.")
else:
    print("\nConversion has nulls, NOT applied. Send me the output first.")

Sample raw values:
['2019-10-01 00:00:00 UTC', '2019-10-01 00:00:03 UTC', '2019-10-01 00:00:07 UTC']

Values NOT matching the expected format: 0

Dtype after conversion: datetime64[us, UTC]
Null after conversion: 0
Date range: 2019-10-01 00:00:00+00:00 -> 2019-12-31 23:59:57+00:00
Rows outside 2019-10: 7877830

Rows per day (first 5 days):
event_time
2019-10-01    137499
2019-10-02    193825
2019-10-03    120452
2019-10-04    111748
2019-10-05    102494
Name: count, dtype: int64

Rows per hour of day:
event_time
0     116075
1     106686
2     117986
3     160528
4     231393
5     354550
6     477039
7     561445
8     622916
9     650949
10    693177
11    711874
12    711702
13    670458
14    626224
15    597539
16    622472
17    668727
18    737533
19    773110
20    696062
21    480582
22    283095
23    162946
Name: count, dtype: int64

Sorted by event_time: True

event_time converted successfully.


In [71]:
# ---- 1. Rows per month (check that no month is incomplete) ----
print("Rows per month:")
print(clean_df["event_time"].dt.strftime("%Y-%m").value_counts().sort_index())

# Rows per event_type per month
print("\nEvent types per month:")
print(pd.crosstab(clean_df["event_time"].dt.strftime("%Y-%m"), clean_df["event_type"]))

# ---- 2. Prepare a copy for CSV export ----
export_df = clean_df.copy()

# Write event_time in the same format as the original data
export_df["event_time"] = export_df["event_time"].dt.strftime("%Y-%m-%d %H:%M:%S UTC")

# Convert category columns to plain strings for CSV
export_df["event_type"] = export_df["event_type"].astype(str)
export_df["brand"] = export_df["brand"].astype(str)

# ---- 3. Save the CSV ----
export_df.to_csv("new_clean_events.csv", index=False)
print("\nSaved: new_clean_events.csv")

# ---- 4. Verify by reading the first rows back ----
check = pd.read_csv("new_clean_events.csv", nrows=5)
print(check)
print(check.dtypes)

# Save the parquet again with the corrected datetime dtype
clean_df.to_parquet("clean_events.parquet", index=False)
print("Saved again: clean_events.parquet")

Rows per month:
event_time
2019-10    3957238
2019-11    4466636
2019-12    3411194
Name: count, dtype: int64

Event types per month:
event_type     cart  purchase  remove_from_cart     view
event_time                                              
2019-10     1213124    245325            642072  1856717
2019-11     1293929    322279            782794  2067634
2019-12      914680    213084            561235  1722195


Task was destroyed but it is pending!
task: <Task pending name='Task-380' coro=<_async_in_context.<locals>.run_in_context() done, defined at C:\Users\yahya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending name='Task-381' coro=<Kernel.shell_main() running at C:\Users\yahya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel\kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at C:\Users\yahya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\zmq\eventloop\zmqstream.py:564]>
C:\Users\yahya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\formats\csvs.py:330: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  libwriters.write_csv_rows(
Task was destroyed but it is pending!
task: <Task pending name='Task-381' coro=<Kernel.shell_main() running at C:\Users\yahya\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ipykernel\kernelbase


Saved: new_clean_events.csv
                event_time event_type  product_id          category_id  \
0  2019-10-01 00:00:00 UTC       cart     5773203  1487580005134238553   
1  2019-10-01 00:00:03 UTC       cart     5773353  1487580005134238553   
2  2019-10-01 00:00:07 UTC       cart     5881589  2151191071051219817   
3  2019-10-01 00:00:07 UTC       cart     5723490  1487580005134238553   
4  2019-10-01 00:00:15 UTC       cart     5881449  1487580013522845895   

    brand  price    user_id                          user_session  \
0  runail   2.62  463240011  26dd6e6e-4dac-4778-8d2c-92e149dab885   
1  runail   2.62  463240011  26dd6e6e-4dac-4778-8d2c-92e149dab885   
2  lovely  13.48  429681830  49e8d843-adf3-428b-a2c3-fe8bc6a307c9   
3  runail   2.62  463240011  26dd6e6e-4dac-4778-8d2c-92e149dab885   
4  lovely   0.56  429681830  49e8d843-adf3-428b-a2c3-fe8bc6a307c9   

   brand_imputed  brand_fallback  
0          False           False  
1          False           False  
2     

In [72]:
# Count rows in the saved CSV in chunks and compare with clean_df
total_rows = 0
for chunk in pd.read_csv("new_clean_events.csv", usecols=["product_id"], chunksize=1_000_000):
    total_rows += len(chunk)

print("Rows in CSV:", total_rows)
print("Rows in clean_df:", len(clean_df))
print("Match:", total_rows == len(clean_df))

Rows in CSV: 11835068
Rows in clean_df: 11835068
Match: True
